<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/RNNBasic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Input
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import EarlyStopping

In [11]:
# 1. Download the IMDB dataset
# 'as_supervised=True' gives us (text, label) pairs directly
dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)

# 2. Split into Train and Validation (Standard 80/20 split)
train_data = dataset['train'].batch(32).prefetch(tf.data.AUTOTUNE)
test_data = dataset['test'].batch(32).prefetch(tf.data.AUTOTUNE)

# 3. Get just the sentences for the .adapt() step
# We take a sample of the text to build our vocabulary
train_sentences = dataset['train'].map(lambda x, y: x)

In [12]:
# 1. Grab 3 examples from the 'train' split
for text, label in dataset['train'].take(3):
    print(f"SENTIMENT: {'Positive' if label == 1 else 'Negative'}")
    print(f"REVIEW: {text.numpy().decode('utf-8')[:250]}...") # Show first 250 characters
    print("-" * 50)

SENTIMENT: Negative
REVIEW: This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline...
--------------------------------------------------
SENTIMENT: Negative
REVIEW: I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa...
--------------------------------------------------
SENTIMENT: Negative
REVIEW: Mann photographs the Alberta Rocky Mountains in a superb fashion, and Jimmy Stewart and Walter Brennan give enjoyable performances as they always seem to do. <br /><br />But come on Hollywood - a Mountie telling the people of Dawson City, Yukon to el...
--------------------------------------------------


In [13]:
# 1. Define the "Resize" for text (e.g., only look at the first 100 words)
max_words = 10000 # Top 10,000 most common words
max_len = 100     # "Resize" every review to 100 words

vectorize_layer = TextVectorization(
    max_tokens=max_words,
    output_mode='int',
    output_sequence_length=max_len
)

# 2. Adapt to your training text (like finding the 'mean' for normalization)
vectorize_layer.adapt(train_sentences)

In [14]:
model_rnn = Sequential([
    Input(shape=(1,), dtype=tf.string), # Input is a raw string
    vectorize_layer,                   # Step 1: Turn string to numbers

    # Step 2: The "Feature Finder" (turns IDs into 64-number vectors)
    Embedding(input_dim=max_words, output_dim=64),

    # Step 3: The "Memory" (The RNN Brick)
    # This is the equivalent of your Conv2D/MaxPool layers!
    SimpleRNN(64),

    # Step 4: The "Head" (Exactly like your CNN!)
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary verdict: Positive or Negative
])

In [15]:
early_stop = EarlyStopping(
    monitor='val_loss',     # 1. Watch the validation loss
    patience=3,              # 2. Wait 3 epochs before giving up
    restore_best_weights=True, # 3. Keep the version that was the 'smartest'
    verbose=1                # 4. Tell us when it stops!
)

In [16]:
model_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# It works exactly like your CNN .fit()!
model_rnn.fit(
    train_data,
    validation_data=test_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 56ms/step - accuracy: 0.5069 - loss: 0.7007 - val_accuracy: 0.5303 - val_loss: 0.6907
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step - accuracy: 0.5622 - loss: 0.6822 - val_accuracy: 0.5499 - val_loss: 0.6821
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.6148 - loss: 0.6526 - val_accuracy: 0.5524 - val_loss: 0.6893
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step - accuracy: 0.7416 - loss: 0.5120 - val_accuracy: 0.5730 - val_loss: 0.7310
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 48ms/step - accuracy: 0.8159 - loss: 0.3908 - val_accuracy: 0.5322 - val_loss: 0.9013
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
